# 02b - Elliptic Bitcoin baselines (ML + GNN)

RF, LightGBM, Temporal-GCN, and GraphSAGE on the Elliptic Bitcoin
dataset (46,564 labeled transactions). Temporal split ts 1-29 (train) /
30-34 (val) / 35-42 (test), matching Weber et al. (2019).

**Outputs:**

- `models/saved/elliptic_rf.joblib` / `elliptic_lgb.joblib`
- `models/saved/elliptic_temporal_gcn.pt` / `elliptic_graphsage.pt`
- `data/processed/elliptic_splits.npz` (X_train/val/test_n, y_*, masks,
  X_all_n, y_all, ts_all)
- `data/processed/elliptic_edge_index.pt`
- `data/processed/elliptic_norm_params.npz`
- `experiments/results/elliptic_feature_names.npy`
- `experiments/results/elliptic_baselines_gnn_results.csv`
- `experiments/figures/elliptic_baselines_gnn.png`,
  `elliptic_feature_importance.png`


In [ ]:
import os
from datetime import datetime

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import (
    compute_metrics, find_optimal_threshold,
    TemporalGCN, GraphSAGEModel, train_gnn, eval_gnn, get_device,
)

set_seed(42)
DEVICE = get_device()

DATA_PATH = str(PATHS.data_raw)
FIGURES_PATH = str(PATHS.figures_dir)
RESULTS_PATH = str(PATHS.results_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [FIGURES_PATH, RESULTS_PATH, MODELS_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'rf': '#2ecc71', 'lgb': '#9b59b6', 'tgcn': '#3498db', 'sage': '#e67e22'}

print('02b - Elliptic baselines (ML + GNN)')
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Device: {DEVICE}")


## 1. Load Elliptic features, labels and edges

Temporal split is derived from `f1` (timestep 1-49). Only
labelled transactions (class 1 = fraud, class 2 = legitimate) are used.


In [ ]:
features_df = pd.read_csv(
    os.path.join(DATA_PATH, 'elliptic_bitcoin_dataset', 'elliptic_txs_features.csv'),
    header=None,
)
classes_df = pd.read_csv(
    os.path.join(DATA_PATH, 'elliptic_bitcoin_dataset', 'elliptic_txs_classes.csv'))
edges_df = pd.read_csv(
    os.path.join(DATA_PATH, 'elliptic_bitcoin_dataset', 'elliptic_txs_edgelist.csv'))

features_df.columns = ['txId'] + [f'f{i}' for i in range(1, features_df.shape[1])]
merged = features_df.merge(classes_df, on='txId')
known = merged[merged['class'].isin(['1', '2', 1, 2])].copy()
known['y'] = known['class'].astype(str).map({'1': 1, '2': 0})
known['ts'] = known['f1'].astype(int)
known = known.sort_values('ts').reset_index(drop=True)

feature_cols = [c for c in known.columns if c.startswith('f') and c != 'f1']
X = known[feature_cols].values.astype(np.float32)
y = known['y'].values
ts = known['ts'].values
print(f"Labelled transactions: {len(known)}, features: {len(feature_cols)}, "
      f"fraud ratio: {y.mean():.4f}, timesteps {ts.min()}..{ts.max()}")

np.save(os.path.join(RESULTS_PATH, 'elliptic_feature_names.npy'), feature_cols)

tx_to_idx = {tx: i for i, tx in enumerate(known['txId'])}
edges = edges_df[edges_df['txId1'].isin(tx_to_idx) & edges_df['txId2'].isin(tx_to_idx)]
edge_index = torch.tensor(
    [[tx_to_idx[e] for e in edges['txId1']],
     [tx_to_idx[e] for e in edges['txId2']]],
    dtype=torch.long,
)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
print(f"Edges (after making undirected): {edge_index.shape[1]}")


## 2. Temporal split and normalization


In [ ]:
train_mask = (ts >= 1) & (ts <= 29)
val_mask = (ts >= 30) & (ts <= 34)
test_mask = (ts >= 35) & (ts <= 42)
X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

mu, sigma = X_train.mean(0), X_train.std(0) + 1e-8
X_train_n = (X_train - mu) / sigma
X_val_n   = (X_val   - mu) / sigma
X_test_n  = (X_test  - mu) / sigma
X_all_n   = (X       - mu) / sigma

np.savez(os.path.join(PROCESSED_PATH, 'elliptic_norm_params.npz'), mu=mu, sigma=sigma)
np.savez(
    os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'),
    X_train=X_train_n, X_val=X_val_n, X_test=X_test_n,
    y_train=y_train, y_val=y_val, y_test=y_test,
    X_all_n=X_all_n, y_all=y, ts_all=ts,
    train_mask=train_mask, val_mask=val_mask, test_mask=test_mask,
)
torch.save(edge_index, os.path.join(PROCESSED_PATH, 'elliptic_edge_index.pt'))

scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"Train {int(train_mask.sum())}  val {int(val_mask.sum())}  test {int(test_mask.sum())}  "
      f"scale_pos_weight={scale_pos_weight:.2f}")


## 3. Train Random Forest and LightGBM


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=2,
    class_weight='balanced', random_state=42, n_jobs=-1,
).fit(X_train_n, y_train)
rf_val = rf.predict_proba(X_val_n)[:, 1]
rf_test = rf.predict_proba(X_test_n)[:, 1]
rf_threshold, _ = find_optimal_threshold(y_val, rf_val)
rf_metrics_val = compute_metrics(y_val, rf_val, rf_threshold)
rf_metrics_test = compute_metrics(y_test, rf_test, rf_threshold)
print(f"RF  val F1={rf_metrics_val['F1']:.4f}  test F1={rf_metrics_test['F1']:.4f}")
joblib.dump(rf, os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))

lgbm = lgb.LGBMClassifier(
    n_estimators=200, max_depth=7, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1,
).fit(X_train_n, y_train)
lgb_val = lgbm.predict_proba(X_val_n)[:, 1]
lgb_test = lgbm.predict_proba(X_test_n)[:, 1]
lgb_threshold, _ = find_optimal_threshold(y_val, lgb_val)
lgb_metrics_val = compute_metrics(y_val, lgb_val, lgb_threshold)
lgb_metrics_test = compute_metrics(y_test, lgb_test, lgb_threshold)
print(f"LGB val F1={lgb_metrics_val['F1']:.4f}  test F1={lgb_metrics_test['F1']:.4f}")
joblib.dump(lgbm, os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))


## 4. Train Temporal-GCN and GraphSAGE

Both models share the training loop from
`xai_blockchain_framework.models.gnn.train_gnn`: capped class weights,
AUC-based early stopping (patience 30), gradient clipping at 1.0, up to
200 epochs at lr=0.01. Thresholds come from the val split and are
re-used on test.


In [ ]:
graph_data = Data(
    x=torch.tensor(X_all_n, dtype=torch.float32),
    y=torch.tensor(y, dtype=torch.long),
    edge_index=edge_index,
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask),
    ts=torch.tensor(ts, dtype=torch.float32),
)

tgcn = TemporalGCN(in_c=len(feature_cols), hid=128, out=2)
tgcn = train_gnn(tgcn, graph_data)
tgcn_val, tgcn_test = eval_gnn(tgcn, graph_data, find_optimal_threshold, compute_metrics)
print(f"T-GCN      val F1={tgcn_val['F1']:.4f}  test F1={tgcn_test['F1']:.4f}")
torch.save(tgcn.state_dict(), os.path.join(MODELS_PATH, 'elliptic_temporal_gcn.pt'))

sage = GraphSAGEModel(in_c=len(feature_cols), hid=128, out=2)
sage = train_gnn(sage, graph_data)
sage_val, sage_test = eval_gnn(sage, graph_data, find_optimal_threshold, compute_metrics)
print(f"GraphSAGE  val F1={sage_val['F1']:.4f}  test F1={sage_test['F1']:.4f}")
torch.save(sage.state_dict(), os.path.join(MODELS_PATH, 'elliptic_graphsage.pt'))


## 5. Summary table and visualizations


In [ ]:
results = []
for name, typ, mv, mt in [
    ('Random Forest', 'ML',  rf_metrics_val,  rf_metrics_test),
    ('LightGBM',      'ML',  lgb_metrics_val, lgb_metrics_test),
    ('Temporal-GCN',  'GNN', tgcn_val,        tgcn_test),
    ('GraphSAGE',     'GNN', sage_val,        sage_test),
]:
    results.append({'Dataset': 'Elliptic', 'Model': name, 'Type': typ, 'Split': 'Val',  **mv})
    results.append({'Dataset': 'Elliptic', 'Model': name, 'Type': typ, 'Split': 'Test', **mt})
results_df = pd.DataFrame(results)
print(results_df[['Model', 'Type', 'Split', 'F1', 'Precision', 'Recall', 'ROC-AUC']].to_string(index=False))
print(f"\nvs Weber et al. (2019):")
print(f"  RF    test F1={rf_metrics_test['F1']:.4f}  ({(rf_metrics_test['F1'] - 0.75) * 100:+.1f}% vs 0.75)")
print(f"  T-GCN test F1={tgcn_test['F1']:.4f}  ({(tgcn_test['F1'] - 0.63) * 100:+.1f}% vs 0.63)")
results_df.to_csv(os.path.join(RESULTS_PATH, 'elliptic_baselines_gnn_results.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
metrics_plot = ['F1', 'Precision', 'Recall', 'ROC-AUC']
x = np.arange(len(metrics_plot)); width = 0.2
for ax_idx, split in enumerate(['Test', 'Val']):
    ax = axes[ax_idx]
    rows = [
        ('RF', results_df[(results_df['Model'] == 'Random Forest') & (results_df['Split'] == split)].iloc[0], COLORS['rf']),
        ('LGB', results_df[(results_df['Model'] == 'LightGBM') & (results_df['Split'] == split)].iloc[0], COLORS['lgb']),
        ('T-GCN', results_df[(results_df['Model'] == 'Temporal-GCN') & (results_df['Split'] == split)].iloc[0], COLORS['tgcn']),
        ('SAGE', results_df[(results_df['Model'] == 'GraphSAGE') & (results_df['Split'] == split)].iloc[0], COLORS['sage']),
    ]
    for i, (lbl, row, col) in enumerate(rows):
        vals = [row[m] for m in metrics_plot]
        ax.bar(x + (i - 1.5) * width, vals, width, label=lbl, color=col)
    ax.set_xticks(x); ax.set_xticklabels(metrics_plot)
    ax.set_ylabel('Score'); ax.set_title(f'Elliptic  {split}')
    ax.set_ylim(0, 1.15); ax.legend(loc='upper right')
    if split == 'Test':
        ax.axhline(0.75, color='red', linestyle='--', alpha=0.5, label='Weber RF')
        ax.axhline(0.63, color='blue', linestyle=':', alpha=0.5, label='Weber GCN')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'elliptic_baselines_gnn.png'), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (model_obj, name, color) in zip(
    axes, [(rf, 'Random Forest', COLORS['rf']), (lgbm, 'LightGBM', COLORS['lgb'])],
):
    imp = pd.DataFrame({'feature': feature_cols, 'importance': model_obj.feature_importances_})
    imp = imp.sort_values('importance', ascending=False).head(15)
    ax.barh(imp['feature'], imp['importance'], color=color)
    ax.set_xlabel('Importance'); ax.set_title(f'{name}  top 15')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'elliptic_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved all Elliptic baseline artifacts (ML + GNN).')
